In [1]:
import sys
import numpy as np
import matplotlib.pyplot as plt
import pickle
sys.path.append('../../map/')
from map import *

In [2]:
with open('/home/tianhui/Intersection_Safety_Challenge_Prediction/Prediction/MTR/output/challenge/mtr+val+ann12_data_withmap_h1s_cluster32/best_result_eval/result.pkl', 'rb') as file:
        val_ann12_h1s_cluster32_best = pickle.load(file)
with open('/home/tianhui/Intersection_Safety_Challenge_Prediction/Prediction/MTR/output/challenge/mtr+val+ann12_data_withmap_h1s_cluster32/val_ann12_h1s_cluster32/eval/eval_with_train/result.pkl', 'rb') as file:
        val_ann12_h1s_cluster32 = pickle.load(file)

In [3]:
len(val_ann12_h1s_cluster32_best)

4839

In [3]:
def WeightedADE(result, num_modes=3):
    avg_weighted_ade = []
    avg_weighted_ade_vehicle = []
    avg_weighted_ade_pedestrian = []
    avg_weighted_ade_cyclist = []
    avg_weighted_ade_dummy = []
    for i in range(len(result)):
        for j in range(len(result[i])):
            pred_trajs = result[i][j]['pred_trajs']
            gt_trajs = result[i][j]['gt_trajs'][11:,:2]
            pred_scores = result[i][j]['pred_scores']
            
            top3_indices = np.argsort(pred_scores)[-num_modes:][::-1]
            top3_pred_trajs = pred_trajs[top3_indices]
            top3_scores = pred_scores[top3_indices]
            normalized_top3_scores = top3_scores / np.sum(top3_scores)

            # weighted ADE caldulation
            t_cut = 11
            T = 91
            weighted_ade = 0

            for k in range(top3_pred_trajs.shape[0]):
                distance = np.sum((gt_trajs - top3_pred_trajs[k])**2, axis=1)
                distance = np.sqrt(distance)
                weighted_ade += normalized_top3_scores[k] * distance

            weighted_ade = np.sum(weighted_ade)
            weighted_ade = weighted_ade / (T - t_cut)
            avg_weighted_ade += [weighted_ade]

            if result[i][j]['object_type'] == 'TYPE_VEHICLE':
                avg_weighted_ade_vehicle += [weighted_ade]
            elif result[i][j]['object_type'] == 'TYPE_PEDESTRIAN':
                avg_weighted_ade_pedestrian += [weighted_ade]
            elif result[i][j]['object_type'] == 'TYPE_CYCLIST':
                avg_weighted_ade_cyclist += [weighted_ade]
            elif result[i][j]['object_type'] == 'TYPE_DUMMY':
                avg_weighted_ade_dummy += [weighted_ade]
         

    avg_weighted_ade = np.mean(avg_weighted_ade)
    avg_weighted_ade_vehicle = np.mean(avg_weighted_ade_vehicle)
    avg_weighted_ade_pedestrian = np.mean(avg_weighted_ade_pedestrian)
    avg_weighted_ade_cyclist = np.mean(avg_weighted_ade_cyclist)
    avg_weighted_ade_dummy = np.mean(avg_weighted_ade_dummy)
    return avg_weighted_ade, avg_weighted_ade_vehicle, avg_weighted_ade_pedestrian, avg_weighted_ade_cyclist, avg_weighted_ade_dummy


In [4]:
# print(WeightedADE(derivheading_fromscratch))
# print('---------')
# print(WeightedADE(val_ann_h1s_cluster64))
# print('---------')
# print(WeightedADE(val_ann_h1s_cluster32))
# print('---------')\
print('\n----best epoch ckpt inference----')
print('avg_weighted_ade, avg_weighted_ade_vehicle, avg_weighted_ade_pedestrian, avg_weighted_ade_cyclist, avg_weighted_ade_dummy')
print('----num_modes = 1-----')
print(WeightedADE(val_ann12_h1s_cluster32_best, num_modes=1))
print('----num_modes = 2-----')
print(WeightedADE(val_ann12_h1s_cluster32_best, num_modes=2))
print('----num_modes = 3-----')
print(WeightedADE(val_ann12_h1s_cluster32_best))

print('\n----epoch 30 ckpt inference----')

print(WeightedADE(val_ann12_h1s_cluster32))
print('----num_modes = 1-----')
print(WeightedADE(val_ann12_h1s_cluster32, num_modes=1))
print('----num_modes = 2-----')
print(WeightedADE(val_ann12_h1s_cluster32, num_modes=2))
print('----num_modes = 3-----')
print(WeightedADE(val_ann12_h1s_cluster32))


----best epoch ckpt inference----
avg_weighted_ade, avg_weighted_ade_vehicle, avg_weighted_ade_pedestrian, avg_weighted_ade_cyclist, avg_weighted_ade_dummy
----num_modes = 1-----
(0.9503066258921676, 2.2345727168557814, 0.7410263090646912, 2.2375545465981066, 0.6287268255258645)
----num_modes = 2-----
(0.9759179326182784, 2.274919813216066, 0.7794422302964337, 2.2520468122989286, 0.6335273716576937)
----num_modes = 3-----
(0.9810410657669443, 2.2867351367281263, 0.7869110963637376, 2.253082735785193, 0.6333785702596592)

----epoch 30 ckpt inference----
(0.9966899757948336, 2.3307897198155034, 0.7912505407591406, 2.5368950425901837, 0.6368387916819882)
----num_modes = 1-----
(0.9731237983510493, 2.293194084284401, 0.7585278222674855, 2.5528828353056388, 0.6277860052048279)
----num_modes = 2-----
(0.9913763749119194, 2.317720093556522, 0.785362433722136, 2.5348321409037036, 0.6351016234316288)
----num_modes = 3-----
(0.9966899757948336, 2.3307897198155034, 0.7912505407591406, 2.53689504

In [4]:
def get_best_mode_byADE(result):
    correct_selections = 0
    total_trajectories = 0
    correct_vehicle = total_vehicle = 0
    correct_pedestrian = total_pedestrian = 0
    correct_cyclist = total_cyclist = 0
    correct_dummy = total_dummy = 0

    for i in range(len(result)):
        for j in range(len(result[i])):
            pred_trajs = result[i][j]['pred_trajs']
            gt_trajs = result[i][j]['gt_trajs'][11:,:2]
            pred_scores = result[i][j]['pred_scores']
            
            # Calculate ADE for each mode
            ade_list = []
            for mode in range(pred_trajs.shape[0]):
                # Calculate the Euclidean distance between the predicted and ground truth trajectories
                distance = np.sum((gt_trajs - pred_trajs[mode])**2, axis=1)
                distance = np.sqrt(distance)
                ade = np.mean(distance)
                ade_list.append(ade)
            # Find the mode with the lowest ADE
            best_ade_index = np.argmin(ade_list)
            # Find the mode with the highest prediction score
            highest_score_index = np.argmax(pred_scores)
            # Check if the mode with the highest score is also the one with the lowest ADE
            if best_ade_index == highest_score_index:
                correct_selections += 1

            total_trajectories += 1

            if result[i][j]['object_type'] == 'TYPE_VEHICLE':
                total_vehicle += 1
                if best_ade_index == highest_score_index:
                    correct_vehicle += 1
            elif result[i][j]['object_type'] == 'TYPE_PEDESTRIAN':
                total_pedestrian += 1
                if best_ade_index == highest_score_index:
                    correct_pedestrian += 1
            elif result[i][j]['object_type'] == 'TYPE_CYCLIST':
                total_cyclist += 1
                if best_ade_index == highest_score_index:
                    correct_cyclist += 1
            elif result[i][j]['object_type'] == 'TYPE_DUMMY':
                total_dummy += 1
                if best_ade_index == highest_score_index:
                    correct_dummy += 1
    # Calculate accuracy as the percentage of correct selections

    accuracy = (correct_selections / total_trajectories) * 100
    accuracy_vehicle = (correct_vehicle / total_vehicle) * 100 if total_vehicle > 0 else 0
    accuracy_pedestrian = (correct_pedestrian / total_pedestrian) * 100 if total_pedestrian > 0 else 0
    accuracy_cyclist = (correct_cyclist / total_cyclist) * 100 if total_cyclist > 0 else 0
    accuracy_dummy = (correct_dummy / total_dummy) * 100 if total_dummy > 0 else 0

    return {
        'overall_accuracy': accuracy,
        'vehicle_accuracy': accuracy_vehicle,
        'pedestrian_accuracy': accuracy_pedestrian,
        'cyclist_accuracy': accuracy_cyclist,
        'dummy_accuracy': accuracy_dummy
    }


In [7]:
get_best_mode_byADE(val_ann12_h1s_cluster32)

{'overall_accuracy': 72.62872628726286,
 'vehicle_accuracy': 64.53305351521512,
 'pedestrian_accuracy': 68.33813315263426,
 'cyclist_accuracy': 51.68539325842697,
 'dummy_accuracy': 82.20706757594544}

In [5]:
import numpy as np

def calculate_minADE(result):
    min_ade_list = []
    min_ade_vehicle = []
    min_ade_pedestrian = []
    min_ade_cyclist = []
    min_ade_dummy = []

    for i in range(len(result)):
        for j in range(len(result[i])):
            pred_trajs = result[i][j]['pred_trajs']
            gt_trajs = result[i][j]['gt_trajs'][11:,:2]

            # Calculate ADE for each mode
            ade_list = []
            for mode in range(pred_trajs.shape[0]):
                distance = np.sum((gt_trajs - pred_trajs[mode])**2, axis=1)
                distance = np.sqrt(distance)
                ade = np.mean(distance)
                ade_list.append(ade)

            # Find the minimum ADE
            min_ade = np.min(ade_list)
            min_ade_list.append(min_ade)

            if result[i][j]['object_type'] == 'TYPE_VEHICLE':
                min_ade_vehicle.append(min_ade)
            elif result[i][j]['object_type'] == 'TYPE_PEDESTRIAN':
                min_ade_pedestrian.append(min_ade)
            elif result[i][j]['object_type'] == 'TYPE_CYCLIST':
                min_ade_cyclist.append(min_ade)
            elif result[i][j]['object_type'] == 'TYPE_DUMMY':
                min_ade_dummy.append(min_ade)

    # Calculate average minADE for each class and overall
    avg_min_ade = np.mean(min_ade_list) if min_ade_list else 0
    avg_min_ade_vehicle = np.mean(min_ade_vehicle) if min_ade_vehicle else 0
    avg_min_ade_pedestrian = np.mean(min_ade_pedestrian) if min_ade_pedestrian else 0
    avg_min_ade_cyclist = np.mean(min_ade_cyclist) if min_ade_cyclist else 0
    avg_min_ade_dummy = np.mean(min_ade_dummy) if min_ade_dummy else 0

    return {
        'overall_minADE': avg_min_ade,
        'vehicle_minADE': avg_min_ade_vehicle,
        'pedestrian_minADE': avg_min_ade_pedestrian,
        'cyclist_minADE': avg_min_ade_cyclist,
        'dummy_minADE': avg_min_ade_dummy
    }

In [6]:
calculate_minADE(val_ann12_h1s_cluster32)

{'overall_minADE': 0.560453873622166,
 'vehicle_minADE': 1.4855780346077505,
 'pedestrian_minADE': 0.47701190355002493,
 'cyclist_minADE': 1.1664228717212193,
 'dummy_minADE': 0.2644170015421655}